In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cpu


In [12]:
block_size = 8
batch_size = 4
max_iters = 10000
#eval_interval = 2500
learning_rate = 3e-4

In [4]:
with open('../data/wizard_of_oz.txt', 'r', encoding='utf-8')as f:
    text = f.read()

chars = sorted(set(text))
print(chars)
vocab_size = len(chars)

['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [5]:
strng_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_strng = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [strng_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_strng[i] for i in l])

# encoded_hello = torch.tensor(encode('hello'),dtype=torch.long)
# decoded_hello = decode(encoded_hello.tolist())

data = torch.tensor(encode(text), dtype=torch.long)

In [16]:
n = int(0.8*len(data))

train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split=='train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    #print(ix)
    x= torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x,y

x,y = get_batch('train')
print('inputs:')
print(x)
print('targets:')
print(y)

inputs:
tensor([[68,  1, 68, 67, 58,  1, 55, 74],
        [68, 67,  1, 58, 75, 58, 71, 78],
        [73,  1, 68, 75, 58, 71,  1, 73],
        [54, 67, 57,  1, 73, 68, 72, 72]])
targets:
tensor([[ 1, 68, 67, 58,  1, 55, 74, 62],
        [67,  1, 58, 75, 58, 71, 78,  1],
        [ 1, 68, 75, 58, 71,  1, 73, 61],
        [67, 57,  1, 73, 68, 72, 72, 58]])


In [17]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('When input is:',context,'target is:',target)

When input is: tensor([28]) target is: tensor(39)
When input is: tensor([28, 39]) target is: tensor(42)
When input is: tensor([28, 39, 42]) target is: tensor(39)
When input is: tensor([28, 39, 42, 39]) target is: tensor(44)
When input is: tensor([28, 39, 42, 39, 44]) target is: tensor(32)
When input is: tensor([28, 39, 42, 39, 44, 32]) target is: tensor(49)
When input is: tensor([28, 39, 42, 39, 44, 32, 49]) target is: tensor(1)
When input is: tensor([28, 39, 42, 39, 44, 32, 49,  1]) target is: tensor(25)


In [18]:
class BigramLanguagemodel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)


    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index) #is 3 dim ->(B,T,C)

        # ---- Compute loss only if targets are provided (training mode) ----
        if targets is None:
            loss = None
        else:
            # Flatten both tensors to feed into cross_entropy
            B,T,C = logits.shape  #T "time" is the sequence size or block_size, C "channel" is vocab_size
            logits = logits.view(B*T, C) # B*T act as total number of samples, C class scores
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets) #cross_Entropy expects input:(N,C), targets:(N,)
            
        return logits,loss

    def generate(self, index, max_new_tokens):
        #index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            #get predictions
            logits, loss = self.forward(index)
            #for generation (not training) targets is None so skips logits flattening -> logtis become (B,T,C)
            logits = logits[:,-1,:] #becomes (B, C), -1 gets only the last token from the time step
            #apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B,C), dim=-1 acts across C class channels
            #sample for distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            #append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1), concatenates across time sequence, if dim=0 it would stack batches
        return index


model = BigramLanguagemodel(vocab_size)
m = model.to(device) 

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


d-B8WJ1HGP0pyXhXc
i WYhh1[S"2[,rG9oeeCMd'YY-z-P*3_ 3x9666JtoP7u,0*cRXTYS5?[d9v(P[667*Q  !qk6haBob2HfjVtQVdBo&lrRLWNlbtAI.Cdi8o6Bk5B P)lvXA84LgEdK;md3qa:(p?Eq7UP*]it0HOSn3o[w-kWoce!Ca6YO9;S"Q8rE6RQGcj3CTzujuTfXelRnB.TY;u)VExnixG9m!U:xqDm5LOUFe!VmvsIhW.65Fb0X8B7A]Q:y5cxwy&MZxoL-dIzY0;MoEqX9f'4Q2Z;"' iUw-k)PuiLpCJbzToP_lDNTdCM]8J[U)H2TnBH;"JSKU5w,*D&dG;uJ1eqmO"j09Huy97N7?[Ieu,*8FEpi 9:(]DE5z8Gu?WYelDT]5Jp9Sn.vESajvQ[(PFqDkqZ;SX6Zc'u:w
!s?TzKbUYrli rmZ4o7266-YCdR0D&**qS"!J2QWCMu[2
XcwSd!4GDaZ4v(J1Qc


In [31]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    #sample a batch of data
    xb,yb = get_batch("train")

    #evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)#make grad star overnot accumulated
    loss.backward()
    optimizer.step()
    
print(loss.item())

2.654493808746338


In [30]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


atha ea RL g anghe nde aveend icch ou, eadring ed n; unk tecar THenon've maind " an d bldithed  the marfrrouroy as anscthinglewod tout abugrde gg
s witon ur soutokincoo co t angere fandeay. cings g wilse aswe he thet wd pealcey y l t Cy to re a ask sonyathe chiz as bloilstabure ven ad aman oy thin uncataminytceme jure, wicks as DPinglo glld ievooul,"

iotherebougo!wed igo an'lw,  t pey ixpodrsthan ghy t sl ry gesa sis.
"Then'se y cemaghe Ild himyoutothertowem."Hulower heam-kin btoly skshoitizad 
